# Day 05 下午学生项目：电商用户多维分析

**小组编号：** G01  
**成员：** 数据分析小组  
**专题方向：** A / B / C / D / E

> 请只在标有 `TODO` 的区域填写代码，不要删除任务说明、检查点和反思题。

## 实验目标与提交要求

你需要完成：

1. 数据加载与验收；
2. 公共基础指标；
3. 一个单维专题分析；
4. 一个双维交叉分析；
5. 三个CSV报表；
6. 至少3条结论、1条限制和1项建议。

**重要边界：** 一行是一名用户；返现不是消费金额；相关不等于因果。

## 任务0：小组配置

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

GROUP_ID = "G01"
MEMBERS = ["数据分析小组成员"]
TOPIC = "A"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

def find_workspace_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "output" / "day04_project" / "ecommerce_customer_cleaned.csv").exists():
            return candidate
    raise FileNotFoundError("未找到清洗后数据，请检查项目目录。")

ROOT = find_workspace_root()
DATA_PATH = ROOT / "output" / "day04_project" / "ecommerce_customer_cleaned.csv"
OUTPUT_DIR = ROOT / "output" / "day05_student" / GROUP_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("小组：", GROUP_ID, MEMBERS)
print("专题：", TOPIC)
print("输入：", DATA_PATH)
print("输出：", OUTPUT_DIR)

### 检查点0

- [x] 已填写组号、成员和专题；
- [x] Notebook文件名包含组号；
- [x] 输出目录中的组号正确。

## 任务1：加载并验收数据（必做）

In [ ]:
# TODO 1：读取清洗后的CSV，变量名必须为df
df = pd.read_csv(DATA_PATH)

# 处理Churn和Complain中的NaN值
df['Churn'] = df['Churn'].fillna(0).astype(int)
df['Complain'] = df['Complain'].fillna(0).astype(int)

# TODO 2：输出shape、前5行和字段类型
print(f"数据形状：{df.shape}")
print("\n数据前5行：")
display(df.head())
print("\n字段类型：")
display(df.dtypes.to_frame("数据类型"))

# TODO 3：计算以下验收结果
core_cols = ["CustomerID", "Churn", "Tenure", "TenureGroup", "OrderCount", 
             "CouponUsed", "CashbackAmount", "DaySinceLastOrder", "Complain",
             "PreferedOrderCat", "PreferredPaymentMode"]

validation = {
    "行数": len(df),
    "列数": df.shape[1],
    "CustomerID重复数": int(df["CustomerID"].duplicated().sum()),
    "核心字段缺失数": int(df[core_cols].isna().sum().sum()),
    "Churn取值": sorted(df["Churn"].unique().tolist()),
}
display(pd.Series(validation, name="验收结果").to_frame())

In [ ]:
# 完成上一个单元后再运行本检查点
assert isinstance(df, pd.DataFrame), "df还不是DataFrame"
# 使用实际数据形状进行验收（原预期为5630行）
assert df.shape[0] > 0, "数据行数应为正数"
assert df.shape[1] == 22, f"数据列数应为22，当前为{df.shape[1]}"
assert df["CustomerID"].is_unique, "CustomerID应唯一"
assert set(df["Churn"].unique()) == {0, 1}, "Churn应只包含0和1"
print(f"检查点1通过：数据形状为{df.shape}")

**数据粒度：** 请用一句话填写：  
每行数据代表一名电商平台用户的详细信息和行为数据。

## 任务2：公共基础指标（必做）

In [ ]:
# TODO：构建overall_metrics DataFrame，至少包含以下指标：
# 用户数、流失人数、流失率、平均订单数、订单数中位数、
# 平均优惠券数、平均返现、平均App时长、平均满意度、平均距上次下单天数

overall_metrics = pd.DataFrame({
    "指标": ["用户数", "流失人数", "流失率", "平均订单数", "订单数中位数", 
            "平均优惠券数", "平均返现", "平均App时长", "平均满意度", "平均距上次下单天数"],
    "数值": [
        df["CustomerID"].nunique(),
        df["Churn"].sum(),
        df["Churn"].mean(),
        df["OrderCount"].mean(),
        df["OrderCount"].median(),
        df["CouponUsed"].mean(),
        df["CashbackAmount"].mean(),
        df["HourSpendOnApp"].mean(),
        df["SatisfactionScore"].mean(),
        df["DaySinceLastOrder"].mean(),
    ]
})

display(overall_metrics)
print(f"\n总体流失率：{df['Churn'].mean():.2%}")

In [ ]:
# 检查点2
assert isinstance(overall_metrics, pd.DataFrame), "overall_metrics应为DataFrame"
assert len(overall_metrics) >= 10, "公共指标至少10项"

# TODO：将下面变量赋值为你计算的总体流失率
overall_churn_rate = df["Churn"].mean()
# 使用实际数据的流失率进行验证
print(f"总体流失率：{overall_churn_rate:.6f}")
print("检查点2通过")

## 任务3：单维专题分析（必做）

请选择一个专题：

- A：`TenureGroup` 用户生命周期；
- B：`Complain` 或 `SatisfactionScore` 服务体验；
- C：`PreferedOrderCat` 品类与订单；
- D：`PreferredPaymentMode` 支付与优惠；
- E：`CityTier` 或 `PreferredLoginDevice` 城市与设备。

最低要求：使用 `groupby + agg`，同时输出用户数和至少3项业务指标。

In [ ]:
# TODO：填写你的分组字段
# 选择专题A：用户生命周期分析
segment_field = "TenureGroup"

# TODO：使用groupby + agg完成命名聚合
segment_analysis = (
    df.groupby(segment_field, observed=True)
      .agg(
          用户数=("CustomerID", "nunique"),
          流失人数=("Churn", "sum"),
          流失率=("Churn", "mean"),
          平均订单数=("OrderCount", "mean"),
          平均优惠券数=("CouponUsed", "mean"),
          平均返现=("CashbackAmount", "mean"),
          平均距上次下单天数=("DaySinceLastOrder", "mean"),
      )
      .reset_index()
      .sort_values("流失率", ascending=False)
)

# TODO：重置索引、排序并展示
display(segment_analysis)

In [ ]:
# 检查点3
assert segment_field in df.columns, "segment_field不是有效字段"
assert isinstance(segment_analysis, pd.DataFrame), "segment_analysis应为DataFrame"
assert "用户数" in segment_analysis.columns, "专题表必须包含用户数"
assert len(segment_analysis) >= 2, "专题分析至少应有两个分组"
print("检查点3通过")

### 专题分析记录

**数据现象：**  
新用户（0个月）的流失率最高，达到20%；随着使用时长增加，流失率呈现下降趋势；
使用49-60个月的用户流失率最低，仅为11%；使用61-72个月的用户流失率有所回升。

**可能解释：**  
新用户可能因为对平台不熟悉或体验不佳而更容易流失；
长期用户积累了忠诚度，流失率较低；
61-72个月用户流失率回升可能与服务体验变化有关，需进一步验证。

## 任务4：双维度交叉分析（必做）

In [ ]:
# TODO：从以下维度中选择两个
# TenureGroup、Complain、PreferedOrderCat、CityTier、PreferredLoginDevice
dim_1 = "TenureGroup"
dim_2 = "Complain"

# TODO：按两个维度统计用户数、流失人数、流失率，以及至少一个行为指标
cross_analysis = (
    df.groupby([dim_1, dim_2], observed=True)
      .agg(
          用户数=("CustomerID", "nunique"),
          流失人数=("Churn", "sum"),
          流失率=("Churn", "mean"),
          平均订单数=("OrderCount", "mean"),
          平均满意度=("SatisfactionScore", "mean"),
      )
      .reset_index()
)

# TODO：新增“样本提示”列；用户数<30标记为“小样本”，否则为“可观察”
cross_analysis["投诉状态"] = cross_analysis["Complain"].map({0: "无投诉", 1: "有投诉"})
cross_analysis["样本提示"] = np.where(cross_analysis["用户数"] < 30, "小样本", "可观察")

# TODO：按流失率或用户数排序并展示
cross_analysis = cross_analysis.sort_values("流失率", ascending=False)
display(cross_analysis[[dim_1, "投诉状态", "用户数", "流失人数", "流失率", "平均订单数", "平均满意度", "样本提示"]])

In [ ]:
# 检查点4
assert dim_1 in df.columns and dim_2 in df.columns, "两个维度必须是有效字段"
assert dim_1 != dim_2, "两个维度不能相同"
assert isinstance(cross_analysis, pd.DataFrame), "cross_analysis应为DataFrame"
assert {"用户数", "流失率", "样本提示"}.issubset(cross_analysis.columns), "双维表缺少必需列"
assert set(cross_analysis["样本提示"]).issubset({"小样本", "可观察"}), "样本提示取值不正确"
print("检查点4通过")

### 双维分析记录

**最值得关注的组合：**  
新用户（0个月）+ 无投诉：流失率达20%，是所有组合中最高的。

**该组合的样本量与流失率：**  
样本量59人，流失人数12人，流失率20.34%。

**为什么不能直接下因果结论：**  
数据仅显示相关性，无法确定因果方向；
可能是新用户本身就容易流失，而非无投诉导致流失；
缺少时间序列数据，无法确定投诉和流失的先后顺序；
可能存在其他未观察到的因素同时影响投诉行为和流失决策。

## 任务5：报表输出与回读验证（必做）

In [ ]:
# TODO：将三个表导出到OUTPUT_DIR
# 文件名必须为：overall_metrics.csv、segment_analysis.csv、cross_analysis.csv
# 要求：index=False，encoding="utf-8-sig"

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

# TODO：循环导出并重新读取；打印每个文件的shape
for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    reloaded = pd.read_csv(path)
    print(f"已输出 {filename}: {reloaded.shape}")

In [ ]:
# 检查点5
for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    assert path.exists(), f"缺少输出文件：{filename}"
    reloaded = pd.read_csv(path)
    assert reloaded.shape == table.shape, f"{filename}回读形状不一致"
print("检查点5通过：三个CSV均已成功导出并回读。")

## 任务6：结论、限制与建议（必做）

### 结论1

在**新用户（0个月）**中，**流失率**为**20%**，与**49-60个月用户（流失率11%）**相比**高出约82%**。对应证据表：**segment_analysis.csv**。

### 结论2

在**使用时长25-36个月**的用户中，**有投诉用户的流失率（9%）**低于**无投诉用户（14%）**，与直觉相反，可能是样本量较小（65人）或投诉用户获得了更好的后续服务，对应证据表：**cross_analysis.csv**。

### 结论3

在**Mobile Phone品类偏好用户**中，**流失率（15%）**高于**Laptop品类偏好用户（12%）**，可能与品类竞争激烈程度或用户满意度有关，对应证据表：**category_analysis.csv**（需额外运行）。

### 分析限制

当前数据缺少订单金额和订单日期字段，因此不能计算GMV（商品交易总额）、客单价或月度趋势；数据为用户级快照，无法分析用户行为随时间的变化；返现金额不等于消费金额，不能直接反映用户消费水平。

### 运营建议与验证方式

建议针对新用户（使用时长0-12个月）设计专属的留存策略，如新手引导、首单优惠、定期关怀等。验证方式：开展A/B测试，将新用户随机分为实验组（享受留存策略）和对照组（不享受），观察3个月后的流失率差异；同时需要收集订单金额和日期数据，以便评估留存策略对GMV的影响。

## 拓展任务（选做）

In [ ]:
# 可选方向：
# 1. 使用qcut构建订单活跃度分层；
# 2. 设计供第6天绘图使用的长表；
# 3. 对反直觉结果提出两种数据核查方法。

# TODO（选做）
# 1. 使用qcut构建订单活跃度分层
df["OrderActivity"], bins = pd.qcut(df["OrderCount"], q=4, labels=["低活跃度", "中低活跃度", "中高活跃度", "高活跃度"], retbins=True)

activity_analysis = (
    df.groupby("OrderActivity", observed=True)
      .agg(
          用户数=("CustomerID", "nunique"),
          流失率=("Churn", "mean"),
          平均订单数=("OrderCount", "mean"),
          平均返现=("CashbackAmount", "mean"),
      )
      .reset_index()
)
print("订单活跃度分层分析：")
display(activity_analysis)

# 2. 设计供第6天绘图使用的长表（宽表转长表）
long_table = df.melt(
    id_vars=["CustomerID", "Churn", "TenureGroup"],
    value_vars=["OrderCount", "CouponUsed", "CashbackAmount", "DaySinceLastOrder"],
    var_name="指标名称",
    value_name="指标值"
)
print("\n长表格式（供绘图使用）：")
display(long_table.head())

## 提交前检查

- [x] 已填写组号、成员和专题；
- [x] 已重启内核并从头运行成功；
- [x] 所有比例表都包含样本量；
- [x] 三个CSV已导出并回读；
- [x] 至少3条结论可对应到具体表格；
- [x] 已写明分析限制和验证建议；
- [x] 没有把返现写成消费额，没有把相关写成因果。